In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip uninstall -y numpy peft accelerate transformers datasets -q

!pip install -q \
  numpy==1.26.4 \
  torch \
  transformers==4.36.2 \
  accelerate==0.25.0 \
  datasets==2.16.1 \
  peft==0.7.1 \
  safetensors==0.4.2 \
  scikit-learn \
  sentencepiece

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.36.2 which is incompatible.
pytensor 2.37.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
tobler 0.13.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-contrib-python 4.13.0.92 requires numpy>

In [ ]:
import os
import torch
import numpy as np
from datasets import load_dataset
from transformers import (
    CLIPProcessor,
    CLIPModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
from transformers.modeling_outputs import SequenceClassifierOutput
from peft import LoraConfig, get_peft_model
from torch import nn
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)


/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
/usr/local/lib/python3.12/dist-packages/transformers/utils/generic.py:309: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(


Using device: cuda


In [ ]:
dataset = load_dataset("anson-huang/mirage-news")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
model_name = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_name)


class CLIPForMFND(nn.Module):
    def __init__(self, model_name, num_labels=2):
        super().__init__()

        base_model = CLIPModel.from_pretrained(model_name)

        # 🔥 LoRA config đúng cho CLIP attention
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            target_modules=[
                "q_proj",
                "v_proj"
            ],
            lora_dropout=0.1,
            bias="none"
        )

        self.clip = get_peft_model(base_model, lora_config)

        self.classifier = nn.Linear(
            base_model.config.projection_dim * 2,
            num_labels
        )

        print("\nTrainable parameters:")
        self.clip.print_trainable_parameters()

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        pixel_values=None,
        labels=None,
    ):
        outputs = self.clip(
            input_ids=input_ids,
            attention_mask=attention_mask,
            pixel_values=pixel_values,
            return_dict=True
        )

        text_embeds = outputs.text_embeds
        image_embeds = outputs.image_embeds

        fused = torch.cat([text_embeds, image_embeds], dim=1)
        logits = self.classifier(fused)

        loss = None
        if labels is not None:
            loss = nn.CrossEntropyLoss()(logits, labels)

        return SequenceClassifierOutput(loss=loss, logits=logits)


model = CLIPForMFND(model_name).to(device)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(



Trainable parameters:
trainable params: 491,520 || all params: 151,768,833 || trainable%: 0.3238609603066527


In [ ]:
def preprocess(example):
    inputs = processor(
        text=example["text"],
        images=example["image"],
        truncation=True,
        padding="max_length",
        max_length=77,
        return_tensors="pt"
    )

    return {
        "input_ids": inputs["input_ids"][0],
        "attention_mask": inputs["attention_mask"][0],
        "pixel_values": inputs["pixel_values"][0],
        "labels": torch.tensor(example["label"], dtype=torch.long)
    }


encoded_dataset = dataset.map(
    preprocess,
    remove_columns=dataset["train"].column_names
)

encoded_dataset.set_format("torch")


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary"
    )

    acc = accuracy_score(labels, preds)
    auc = roc_auc_score(labels, probs[:, 1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "auc": auc
    }


In [ ]:
output_dir = "/content/drive/MyDrive/CLIP_LoRA_MirageNews"
os.makedirs(output_dir, exist_ok=True)


training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    learning_rate=2e-5,
    weight_decay=0.01,

    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="f1",

    fp16=torch.cuda.is_available(),
    report_to="none"
)


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/accelerate/accelerator.py:439: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Auc
1,0.644000,0.397645,0.988000,0.981073,0.995200,0.988086,0.997801
2,0.369600,0.240031,0.995600,0.995997,0.995200,0.995598,0.999431
3,0.250200,0.183305,0.995200,0.992051,0.998400,0.995215,0.999627
4,0.170300,0.158899,0.996800,0.995215,0.998400,0.996805,0.999748
5,0.158700,0.151778,0.996800,0.995215,0.998400,0.996805,0.999752


TrainOutput(global_step=3125, training_loss=0.2926027185058594, metrics={'train_runtime': 1258.3787, 'train_samples_per_second': 39.734, 'train_steps_per_second': 2.483, 'total_flos': 0.0, 'train_loss': 0.2926027185058594, 'epoch': 5.0})

In [ ]:
metrics = trainer.evaluate()
print(metrics)


{'eval_loss': 0.15889932215213776, 'eval_accuracy': 0.9968, 'eval_precision': 0.9952153110047847, 'eval_recall': 0.9984, 'eval_f1': 0.9968051118210862, 'eval_auc': 0.99974848, 'eval_runtime': 34.858, 'eval_samples_per_second': 71.719, 'eval_steps_per_second': 2.266, 'epoch': 5.0}


In [ ]:
model.clip.save_pretrained(output_dir)
processor.save_pretrained(output_dir)

torch.save(
    model.classifier.state_dict(),
    os.path.join(output_dir, "classifier.pt")
)

print("Model saved to:", output_dir)


Model saved to: /content/drive/MyDrive/CLIP_LoRA_MirageNews


In [ ]:
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())

print(f"Trainable params: {trainable:,}")
print(f"Total params: {total:,}")
print(f"Trainable %: {100 * trainable / total:.4f}%")

Trainable params: 493,570
Total params: 151,770,883
Trainable %: 0.3252%
